# Gasoil Flat Price Contracts — Open Interest EDA

Analyze the relative size of each Gasoil flat price contract.

Gasoil positioning comes from two independent sources:
- **ICE Futures Europe** — the primary benchmark contract (Bloomberg's QS ticker)
- **CFTC (NYMEX)** — cash-settled swaps and options referencing ICE Gasoil

### ICE Europe
- ICE Gasoil Futures and Options - ICE Futures Europe

### CFTC Flat price codes (from `docs/gasoil_cot_mapping.md`):
- `02265V` — GASOIL (ICE) SWAP - NEW YORK MERCANTILE EXCHANGE
- `022A46` — GASOIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE
- `022A24` — GASOIL (ICE) MINI CAL SWAP - NEW YORK MERCANTILE EXCHANGE
- `02265J` — SING GASOIL SWAP - NEW YORK MERCANTILE EXCHANGE
- `022A22` — SING GASOIL BALMO SWAP - NEW YORK MERCANTILE EXCHANGE

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## Load CFTC Data

In [2]:
df_cftc = pd.read_csv("../../downloads/cftc/disaggregated_combined.csv", low_memory=False)

GASOIL_FLAT_PRICE_CODES = [
    "02265V",  # GASOIL (ICE) SWAP - NYMEX
    "022A46",  # GASOIL AVG PRICE OPTIONS - NYMEX
    "022A24",  # GASOIL (ICE) MINI CAL SWAP - NYMEX (added per Marouen)
    "02265J",  # SING GASOIL SWAP - NYMEX (added per Marouen)
    "022A22",  # SING GASOIL BALMO SWAP - NYMEX (added per Marouen)
]

cftc = df_cftc[df_cftc["cftc_contract_market_code"].isin(GASOIL_FLAT_PRICE_CODES)].copy()
cftc["report_date"] = pd.to_datetime(cftc["report_date_as_yyyy_mm_dd"])
cftc["open_interest_all"] = pd.to_numeric(cftc["open_interest_all"], errors="coerce")
cftc["source"] = "CFTC (NYMEX)"
cftc["label"] = cftc["cftc_contract_market_code"] + " — " + cftc["market_and_exchange_names"]
print(f"CFTC Gasoil flat price rows: {len(cftc):,}")
print(f"Codes found: {sorted(cftc['cftc_contract_market_code'].unique())}")

CFTC Gasoil flat price rows: 409
Codes found: ['02265J', '02265V', '022A22', '022A24', '022A46']


## Load ICE Europe Data

In [3]:
df_ice = pd.read_csv("../../downloads/ice/ice_cot.csv", low_memory=False)

ice = df_ice[df_ice["Market_and_Exchange_Names"].str.contains("Gasoil", case=False, na=False)].copy()
ice["report_date"] = pd.to_datetime(ice["As_of_Date_Form_MM/DD/YYYY"])
ice["open_interest_all"] = pd.to_numeric(ice["Open_Interest_All"], errors="coerce")
ice["cftc_contract_market_code"] = "ICE_GASOIL"
ice["market_and_exchange_names"] = ice["Market_and_Exchange_Names"]
ice["source"] = "ICE Europe"
ice["label"] = "ICE_GASOIL — " + ice["Market_and_Exchange_Names"]
print(f"ICE Gasoil rows: {len(ice):,}")
print(f"Contract names: {sorted(ice['Market_and_Exchange_Names'].unique())}")

ICE Gasoil rows: 1,590
Contract names: ['ICE Gasoil Futures - ICE Futures Europe', 'ICE Gasoil Futures and Options - ICE Futures Europe']


## Combine All Gasoil Flat Price Contracts

In [4]:
cols = ["report_date", "cftc_contract_market_code", "market_and_exchange_names", "open_interest_all", "source", "label"]
gasoil = pd.concat([cftc[cols], ice[cols]], ignore_index=True)
print(f"Total Gasoil flat price rows: {len(gasoil):,}")
print(f"Date range: {gasoil['report_date'].min().date()} to {gasoil['report_date'].max().date()}")

Total Gasoil flat price rows: 1,999
Date range: 2008-03-11 to 2026-03-24


## Average Open Interest by Contract

In [5]:
code_labels = (
    gasoil.groupby("cftc_contract_market_code")["market_and_exchange_names"]
    .first()
    .to_dict()
)

source_labels = (
    gasoil.groupby("cftc_contract_market_code")["source"]
    .first()
    .to_dict()
)

avg_oi = (
    gasoil.groupby("cftc_contract_market_code")["open_interest_all"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("avg_open_interest")
)
avg_oi["contract_name"] = avg_oi.index.map(code_labels)
avg_oi["source"] = avg_oi.index.map(source_labels)
avg_oi["pct_of_total"] = (avg_oi["avg_open_interest"] / avg_oi["avg_open_interest"].sum() * 100)
avg_oi["avg_open_interest"] = avg_oi["avg_open_interest"].round(0).astype(int)
avg_oi["pct_of_total"] = avg_oi["pct_of_total"].round(2)

avg_oi[["contract_name", "source", "avg_open_interest", "pct_of_total"]]

,contract_name,source,avg_open_interest,pct_of_total
cftc_contract_market_code,,,,
ICE_GASOIL,ICE Gasoil Futures - ICE Futures Europe,ICE Europe,776091,96.20
02265J,SING GASOIL SWAP - NEW YORK MERCANTILE EXCHANGE,CFTC (NYMEX),15813,1.96
022A24,GASOIL (ICE) MINI CAL SWAP - NEW YORK MERCANTI...,CFTC (NYMEX),4914,0.61
02265V,GASOIL (ICE) SWAP - NEW YORK MERCANTILE EXCHANGE,CFTC (NYMEX),4375,0.54
022A46,GASOIL AVG PRICE OPTIONS - NEW YORK MERCANTILE...,CFTC (NYMEX),3490,0.43
022A22,SING GASOIL BALMO SWAP - NEW YORK MERCANTILE E...,CFTC (NYMEX),2033,0.25


## Last Report Date by Contract Code

In [6]:
last_report = (
    gasoil.groupby("cftc_contract_market_code")
    .agg(
        contract_name=("market_and_exchange_names", "first"),
        source=("source", "first"),
        last_report_date=("report_date", "max"),
        last_report_oi=("open_interest_all", "last"),
    )
    .sort_values("last_report_date", ascending=False)
)
last_report["last_report_date"] = last_report["last_report_date"].dt.date
last_report

,contract_name,source,last_report_date,last_report_oi
cftc_contract_market_code,,,,
ICE_GASOIL,ICE Gasoil Futures - ICE Futures Europe,ICE Europe,2026-03-24,902564.0
02265J,SING GASOIL SWAP - NEW YORK MERCANTILE EXCHANGE,CFTC (NYMEX),2017-08-22,17364.0
02265V,GASOIL (ICE) SWAP - NEW YORK MERCANTILE EXCHANGE,CFTC (NYMEX),2012-09-04,3352.0
022A22,SING GASOIL BALMO SWAP - NEW YORK MERCANTILE E...,CFTC (NYMEX),2012-07-31,1526.0
022A24,GASOIL (ICE) MINI CAL SWAP - NEW YORK MERCANTI...,CFTC (NYMEX),2011-09-27,3850.0
022A46,GASOIL AVG PRICE OPTIONS - NEW YORK MERCANTILE...,CFTC (NYMEX),2010-12-28,3539.0


## Open Interest Share — Pie Chart

In [7]:
fig = px.pie(
    avg_oi.reset_index(),
    values="avg_open_interest",
    names="cftc_contract_market_code",
    hover_data=["contract_name"],
    title="Gasoil Flat Price Contracts — Average Open Interest Share",
)
fig.update_traces(textinfo="label+percent", textposition="outside")
fig.show()

## Open Interest Over Time by Contract

In [8]:
fig = px.line(
    gasoil.sort_values("report_date"),
    x="report_date",
    y="open_interest_all",
    color="cftc_contract_market_code",
    hover_data=["market_and_exchange_names"],
    title="Gasoil Flat Price Contracts — Open Interest Over Time",
    labels={
        "report_date": "Report Date",
        "open_interest_all": "Open Interest",
        "cftc_contract_market_code": "Contract Code",
    },
)
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.3))
fig.show()

## Open Interest Over Time — Stacked Area (Percentage)

In [9]:
pivot = gasoil.pivot_table(
    index="report_date",
    columns="cftc_contract_market_code",
    values="open_interest_all",
    aggfunc="sum",
).fillna(0)

pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig = go.Figure()
for col in pct.columns:
    label = code_labels.get(col, col)
    fig.add_trace(go.Scatter(
        x=pct.index, y=pct[col],
        name=col,
        hovertext=label,
        stackgroup="one",
        groupnorm="percent",
    ))
fig.update_layout(
    title="Gasoil Flat Price Contracts — OI Share Over Time (%)",
    xaxis_title="Report Date",
    yaxis_title="% of Total Gasoil Flat Price OI",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3),
)
fig.show()

## Summary Table — Latest Reporting Date

In [10]:
latest_date = gasoil["report_date"].max()
latest = gasoil[gasoil["report_date"] == latest_date].copy()

summary = (
    latest.groupby("cftc_contract_market_code")
    .agg(
        contract_name=("market_and_exchange_names", "first"),
        open_interest=("open_interest_all", "sum"),
    )
    .sort_values("open_interest", ascending=False)
)
summary["pct_of_total"] = (summary["open_interest"] / summary["open_interest"].sum() * 100).round(2)

print(f"Latest report date: {latest_date.date()}")
summary

Latest report date: 2026-03-24


,contract_name,open_interest,pct_of_total
cftc_contract_market_code,,,
ICE_GASOIL,ICE Gasoil Futures and Options - ICE Futures E...,1956087.0,100.0
